# OmniParser v2 — Interactive Parser (Google Colab T4)

This notebook mirrors `parser.py` and handles one-time setup (repo clone + weight
download, same logic as `setup_weights.py`) so you can run it end-to-end on Colab.

**Runtime:** Runtime → Change runtime type → T4 GPU.

Working directory on Colab is `/content`, so everything lands at:

```
/content/
├── OmniParser/              # cloned microsoft/OmniParser repo
├── weights/
│   ├── icon_detect/model.pt
│   └── icon_caption_florence/...
└── output/                  # annotated PNG + parsed JSON
```

## 1. Install dependencies

Colab's base image is missing `easyocr`, `ultralytics`, and a few others that
`OmniParser/util/utils.py` imports at module load.

In [1]:
%pip install -q \
    "transformers==4.49.0" \
    "ultralytics==8.3.81" \
    "supervision==0.18.0" \
    "einops==0.8.0" \
    timm easyocr huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 123.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.1/922.1 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 34.2 MB/s eta 0:00:00


## 2. Config

Tweak these and re-run from here down.

In [15]:
from pathlib import Path

DRIVE_PATH = Path("/content/drive/MyDrive/ContextDrive/")
IMAGE_PATH = DRIVE_PATH / "UISampleData" / "images" / "sample_1.png"
OUTPUT_DIR = DRIVE_PATH / "UISampleData" / "outputs"
BOX_THRESHOLD = 0.05
DEVICE_OVERRIDE: str | None = None  # "cuda" | "cpu" | None (auto -> cuda on T4)

HERE = Path.cwd().resolve()
OMNIPARSER_DIR = HERE / "OmniParser"
WEIGHTS_DIR = HERE / "weights"
YOLO_WEIGHTS = WEIGHTS_DIR / "icon_detect" / "model.pt"
CAPTION_WEIGHTS_DIR = WEIGHTS_DIR / "icon_caption_florence"

print(f"HERE: {HERE}")
print(f"OmniParser repo present: {OMNIPARSER_DIR.is_dir()}")
print(f"YOLO weights present:    {YOLO_WEIGHTS.is_file()}")
print(f"Caption weights present: {CAPTION_WEIGHTS_DIR.is_dir()}")

HERE: /content
OmniParser repo present: True
YOLO weights present:    True
Caption weights present: True


In [3]:
import glob
import os

# List all files in current directory
files = glob.glob("*")
print("Files in current directory:")
for file in files:
    print(file)

Files in current directory:
sample_data


## 3. Clone OmniParser repo (if missing)

`parser.py` imports from `OmniParser/util/utils.py`; if the repo isn't next to us, clone it.

In [4]:
import subprocess

if not OMNIPARSER_DIR.is_dir():
    print("cloning microsoft/OmniParser...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/microsoft/OmniParser.git",
        str(OMNIPARSER_DIR),
    ])
else:
    print(f"OmniParser repo already at {OMNIPARSER_DIR}")

cloning microsoft/OmniParser...


## 4. Download weights (if missing)

Pulls `icon_detect/model.pt` and the Florence-2 caption model from `microsoft/OmniParser-v2.0`.
Skips files that already exist.

In [5]:
from huggingface_hub import hf_hub_download, snapshot_download

REPO_ID = "microsoft/OmniParser-v2.0"


def download_yolo() -> None:
    target_dir = WEIGHTS_DIR / "icon_detect"
    target_dir.mkdir(parents=True, exist_ok=True)
    for fname in ("model.pt", "model.yaml", "train_args.yaml"):
        try:
            hf_hub_download(
                repo_id=REPO_ID,
                filename=f"icon_detect/{fname}",
                local_dir=str(WEIGHTS_DIR),
            )
        except Exception as e:
            if fname == "model.pt":
                raise
            print(f"skip optional {fname}: {e}")
    print(f"yolo weights ready at {target_dir}")


def download_caption_model() -> None:
    target_dir = WEIGHTS_DIR / "icon_caption_florence"
    target_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=REPO_ID,
        allow_patterns=["icon_caption/*"],
        local_dir=str(WEIGHTS_DIR),
    )
    src = WEIGHTS_DIR / "icon_caption"
    if src.is_dir() and not any(target_dir.iterdir()):
        for child in src.iterdir():
            child.rename(target_dir / child.name)
        src.rmdir()
    print(f"caption weights ready at {target_dir}")


WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

if not YOLO_WEIGHTS.is_file():
    download_yolo()
else:
    print(f"yolo weights already at {YOLO_WEIGHTS}")

if not CAPTION_WEIGHTS_DIR.is_dir() or not any(CAPTION_WEIGHTS_DIR.iterdir()):
    download_caption_model()
else:
    print(f"caption weights already at {CAPTION_WEIGHTS_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


icon_detect/model.pt:   0%|          | 0.00/40.6M [00:00<?, ?B/s]

model.yaml: 0.00B [00:00, ?B/s]

train_args.yaml: 0.00B [00:00, ?B/s]

yolo weights ready at /content/weights/icon_detect


...

config.json: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

icon_caption/model.safetensors:   0%|          | 0.00/1.08G [00:00<?, ?B/s]

caption weights ready at /content/weights/icon_caption_florence


## 5. Imports + paddleocr stub

`OmniParser/util/utils.py` imports `paddleocr` at module-load time, but we use the
easyocr path only. Stubbing it out avoids pulling in the heavy dep on Colab.

In [6]:
import base64
import json
import sys
import types

import torch
from PIL import Image

if str(OMNIPARSER_DIR) not in sys.path:
    sys.path.insert(0, str(OMNIPARSER_DIR))

if "paddleocr" not in sys.modules:
    _paddle_stub = types.ModuleType("paddleocr")

    class _PaddleOCRStub:
        def __init__(self, *args, **kwargs):
            pass

    _paddle_stub.PaddleOCR = _PaddleOCRStub
    sys.modules["paddleocr"] = _paddle_stub

from util.utils import (
    check_ocr_box,
    get_caption_model_processor,
    get_som_labeled_img,
    get_yolo_model,
)

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Completetorch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


## 6. Device + model loaders

On Colab T4 we want CUDA + fp16 for the Florence-2 caption model — that's what
`get_caption_model_processor` already does when `device != "cpu"`.

In [7]:
def pick_device(override: str | None = None) -> str:
    if override:
        return override
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def load_models(device: str):
    if not YOLO_WEIGHTS.is_file():
        raise SystemExit(f"missing YOLO weights at {YOLO_WEIGHTS}")
    if not CAPTION_WEIGHTS_DIR.is_dir():
        raise SystemExit(f"missing caption weights at {CAPTION_WEIGHTS_DIR}")

    yolo = get_yolo_model(model_path=str(YOLO_WEIGHTS))
    if device == "cuda":
        yolo = yolo.to("cuda")
    caption = get_caption_model_processor(
        model_name="florence2",
        model_name_or_path=str(CAPTION_WEIGHTS_DIR),
        device=device,
    )
    return yolo, caption


device = pick_device(DEVICE_OVERRIDE)
print(f"device: {device}")
yolo_model, caption_model_processor = load_models(device)
print("models loaded")

device: cuda
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

...

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


models loaded


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAMPLE_FOLDER = Path("/content/drive/MyDrive/ContextDrive/UISampleData/")

## 7. Parse an image

Upload a screenshot to `/content/` (or change `IMAGE_PATH` above) before running.

In [ ]:
def parse_image(
    image_path: Path,
    output_dir: Path,
    yolo_model,
    caption_model_processor,
    box_threshold: float,
):
    image = Image.open(image_path)
    box_overlay_ratio = max(image.size) / 3200
    draw_bbox_config = {
        "text_scale": 0.8 * box_overlay_ratio,
        "text_thickness": max(int(2 * box_overlay_ratio), 1),
        "text_padding": max(int(3 * box_overlay_ratio), 1),
        "thickness": max(int(3 * box_overlay_ratio), 1),
    }

    (text, ocr_bbox), _ = check_ocr_box(
        str(image_path),
        display_img=False,
        output_bb_format="xyxy",
        goal_filtering=None,
        easyocr_args={"paragraph": True, "text_threshold": 0.9},
        use_paddleocr=False,
    )

    labeled_img_b64, _label_coords, parsed_content = get_som_labeled_img(
        str(image_path),
        yolo_model,
        BOX_TRESHOLD=box_threshold,
        output_coord_in_ratio=False,
        ocr_bbox=ocr_bbox,
        draw_bbox_config=draw_bbox_config,
        caption_model_processor=caption_model_processor,
        ocr_text=text,
        use_local_semantics=True,
        iou_threshold=0.7,
        imgsz=640,
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem
    annotated_path = output_dir / f"{stem}_annotated.png"
    annotated_path.write_bytes(base64.b64decode(labeled_img_b64))

    json_path = output_dir / f"{stem}_parsed.json"
    json_path.write_text(json.dumps(parsed_content, indent=2))

    return annotated_path, json_path, labeled_img_b64, parsed_content


image_path = IMAGE_PATH.expanduser().resolve()
if not image_path.is_file():
    raise SystemExit(f"image not found: {image_path}")

output_dir = OUTPUT_DIR.expanduser().resolve()

annotated_path, json_path, labeled_img_b64, parsed_content = parse_image(
    image_path, output_dir, yolo_model, caption_model_processor, BOX_THRESHOLD
)
print(f"wrote {annotated_path}")
print(f"wrote {json_path}")


0: 736x1280 151 icons, 43.3ms
Speed: 8.1ms preprocess, 43.3ms inference, 1.7ms postprocess per image at shape (1, 3, 736, 1280)
len(filtered_boxes): 253 177
time to get parsed content: 0.410383939743042
wrote /content/drive/MyDrive/ContextDrive/UISampleData/outputs/sample_1_annotated.png
wrote /content/drive/MyDrive/ContextDrive/UISampleData/outputs/sample_1_parsed.json


## 8. Preview results inline

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display

display(IPyImage(data=base64.b64decode(labeled_img_b64)))

print(f"{len(parsed_content)} elements parsed")
for i, el in enumerate(parsed_content[:20]):
    print(f"  [{i}] {el}")
if len(parsed_content) > 20:
    print(f"  ... and {len(parsed_content) - 20} more")